# Colab Session: repro_s1_lab09
Generated from colab-cli history log.

In [ ]:
# @title [Environment & Reproducibility Metadata]
import sys, platform, torch, transformers
print("=== Reproducibility Environment Info ===")
print("Python version      :", sys.version.split()[0])
print("Platform            :", platform.platform())
print("PyTorch version     :", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device          :", torch.cuda.get_device_name(0))
    print("CUDA version        :", torch.version.cuda)
print("Default Random Seed : 42 (random.seed(42), torch.manual_seed(42))")
print("Stage 1 Model Dtype : torch.bfloat16")


=== Reproducibility Environment Info ===
Python version      : 3.13.16
Platform            : Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch version     : 2.11.0+cu130
Transformers version: 5.20.2
CUDA available      : True
GPU Device          : NVIDIA L4
CUDA version        : 13.0
Default Random Seed : 42 (random.seed(42), torch.manual_seed(42))
Stage 1 Model Dtype : torch.bfloat16


In [ ]:
# ==============================================================================
# ORIGINAL EXECUTED SCRIPT EXTRACTED VERBATIM FROM TRANSCRIPT (AST-UNESCAPED)
# Transcript Step Index : 418
# Timestamp             : 2026-10-30T06:52:34Z
# Original Colab Session: stage1_stress_bench
# ==============================================================================

import os, sys, time, math, random, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F

if not os.path.exists('/content/minimind'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jingyaogong/minimind.git', '/content/minimind'], check=True)
sys.path.insert(0, '/content/minimind')

from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

tokenizer = AutoTokenizer.from_pretrained('/content/minimind/model')
MASK_ID = 3
SLOT_PAD_ID = 4
BOS_ID = tokenizer.bos_token_id
EOS_ID = tokenizer.eos_token_id

# ==============================================================================
# NON-SATURATED 40-TASK STRESS BENCHMARK
# Tier 1 (16 tasks): High-Entropy Confusable Serial Numbers (differ by 1 char!)
# Tier 2 ( 8 tasks): Multi-Law Zero-Shot 1-Hop Analogy (4 Unseen Domains x 2 Competing Laws)
# Tier 3 ( 8 tasks): Long-Horizon 3-Hop Zero-Shot Chain (Tests Latent Drift at Hop 3!)
# Tier 4 ( 8 tasks): Hybrid Compound Tasks (Requires BOTH Unseen Analogy + Exact Emergency Code!)
# ==============================================================================

# 16 Highly Confusable Rote Serial Numbers (4 clusters x 4 near-identical nodes)
rote_serials = [
    ("查询节点A1密钥：", "SN-8841-902-A1"),
    ("查询节点A2密钥：", "SN-8841-902-A2"),
    ("查询节点A3密钥：", "SN-8841-902-A3"),
    ("查询节点A4密钥：", "SN-8841-902-A4"),
    ("查询节点B1密钥：", "SN-7739-105-B1"),
    ("查询节点B2密钥：", "SN-7739-105-B2"),
    ("查询节点B3密钥：", "SN-7739-105-B3"),
    ("查询节点B4密钥：", "SN-7739-105-B4"),
    ("查询节点C1密钥：", "SN-6628-441-C1"),
    ("查询节点C2密钥：", "SN-6628-441-C2"),
    ("查询节点C3密钥：", "SN-6628-441-C3"),
    ("查询节点C4密钥：", "SN-6628-441-C4"),
    ("查询节点D1密钥：", "SN-5517-339-D1"),
    ("查询节点D2密钥：", "SN-5517-339-D2"),
    ("查询节点D3密钥：", "SN-5517-339-D3"),
    ("查询节点D4密钥：", "SN-5517-339-D4"),
]

# Domain Emergency Codes for Tier 4 Hybrid Tasks
domain_emg_codes = [
    ("查询人体医学急救码：", "EMG-MED-119"),
    ("查询网络通讯抢修码：", "EMG-NET-404"),
    ("查询工厂产线维修码：", "EMG-FAC-502"),
    ("查询呼吸系统急救码：", "EMG-AIR-999"),
]

# 6 Domains x 2 Competing Laws x 4 Stages (s0 -> s1 -> s2 -> s3)
# Law 0: Narrowing -> Pressure Surge -> Blockage -> Relief Valve
# Law 1: Supply Cut -> Reserve Drain -> Starvation -> Backup Supply
multi_law_domains = [
    ("水利工程", True, {
        0: ["水流在水管稳定流动，", "当水管内壁结垢变窄时，", "管内水压就会急剧升高，", "最终导致水流堵塞中断。"],
        1: ["水库向管网正常供水，", "当上游水源突然断流时，", "管网储水就会迅速枯竭，", "最终导致末端供水停摆。"]
    }),
    ("交通工程", True, {
        0: ["车流在高速路正常行驶，", "当施工导致车道变窄时，", "路段交通压力急剧升高，", "最终导致车流瘫痪堵塞。"],
        1: ["巴士向干线正常发车，", "当总站发车突然中断时，", "干线运力就会迅速枯竭，", "最终导致站点运输停摆。"]
    }),
    ("人體醫學", False, {
        0: ["血液在血管中正常循环，", "当血管壁堆积胆固醇变窄时，", "血管内血压就会急剧升高，", "最终导致血液循环受阻。"],
        1: ["心脏向全身正常供血，", "当心室供血突然中断时，", "器官血氧就会迅速枯竭，", "最终导致组织机能停摆。"]
    }),
    ("網路通訊", False, {
        0: ["数据在光纤中高速传输，", "当网络带宽通道受限变窄时，", "网络延迟与拥塞急剧升高，", "最终导致数据包传输中断。"],
        1: ["核心网向基站正常发包，", "当主干路由突然断连时，", "基站缓存就会迅速枯竭，", "最终导致通信服务停摆。"]
    }),
    ("工廠產線", False, {
        0: ["零件在流水线顺畅运转，", "当关键工位故障产能变窄时，", "产线积压負荷急剧升高，", "最终导致工厂生产停摆。"],
        1: ["仓库向产线正常供料，", "当上游原料突然断供时，", "产线库存就会迅速枯竭，", "最终导致装配作业停摆。"]
    }),
    ("呼吸系統", False, {
        0: ["空气在气管中顺畅进出，", "当支气管发炎肿胀变窄时，", "呼吸道阻力就会急剧升高，", "最终导致呼吸困难窒息。"],
        1: ["肺部向血液正常供氧，", "当外界氧气突然断绝时，", "体内血氧就会迅速枯竭，", "最终导致呼吸代谢停摆。"]
    }),
]

SLOT_LEN = 12
PROMPT_LEN = 16

def pad_slot(txt, length=SLOT_LEN):
    ids = tokenizer(txt, add_special_tokens=False).input_ids[:length]
    val = [1]*len(ids) + [0]*(length - len(ids))
    pad = ids + [SLOT_PAD_ID]*(length - len(ids))
    return torch.tensor([pad], device='cuda'), torch.tensor([val], device='cuda')

def pad_prompt(txt, length=PROMPT_LEN):
    ids = [BOS_ID] + tokenizer(txt, add_special_tokens=False).input_ids[:length - 1]
    ids = [SLOT_PAD_ID]*(length - len(ids)) + ids
    return torch.tensor([ids], device='cuda')

def clean_str(t_ids):
    c = [tid for tid in t_ids.tolist() if tid not in (MASK_ID, SLOT_PAD_ID, BOS_ID, EOS_ID)]
    return tokenizer.decode(c)

all_rote = rote_serials + domain_emg_codes
rote_tensors = [(pad_prompt(q), pad_slot(a)[0], q, a) for q, a in all_rote]

# Structure states: 2 Laws x 4 Stages = 8 distinct universal structural states!
# Domain states: 6 distinct domains!
dom_law_tensors = []
for d_idx, (d_name, is_seen, laws_dict) in enumerate(multi_law_domains):
    for law_id in (0, 1):
        clauses = laws_dict[law_id]
        c_t = [pad_slot(c) for c in clauses]
        dom_law_tensors.append((d_idx, d_name, is_seen, law_id, clauses, c_t))

class PureDiffusionModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = MiniMindForCausalLM(config)
        for layer in self.backbone.model.layers:
            layer.self_attn.is_causal = False

    def forward_loss(self, prompt_ids, target_slots):
        B, L = target_slots.shape
        m = (torch.rand((B, L), device=target_slots.device) < 0.6)
        m[:, 0] = True
        canv = target_slots.clone(); canv[m] = MASK_ID
        lbl = target_slots.clone(); lbl[~m] = -100
        full_in = torch.cat([prompt_ids, canv], dim=1)
        logits = self.backbone(full_in).logits[:, prompt_ids.size(1):, :].contiguous()
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), lbl.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate_slots(self, prompt_ids, steps=2):
        B = prompt_ids.size(0)
        canv = torch.full((B, SLOT_LEN), MASK_ID, dtype=torch.long, device=prompt_ids.device)
        for s in range(steps):
            logits = self.backbone(torch.cat([prompt_ids, canv], dim=1)).logits[:, prompt_ids.size(1):, :]
            conf, pred = F.softmax(logits, dim=-1).max(dim=-1)
            still_m = (canv == MASK_ID)
            n_rem = still_m[0].sum().item()
            if n_rem == 0: break
            conf = conf.masked_fill(~still_m, -1.0)
            k = n_rem if s == steps - 1 else max(1, SLOT_LEN // steps)
            _, top_idx = torch.topk(conf, k=min(k, n_rem), dim=-1)
            for b in range(B):
                for idx in top_idx[b]:
                    if still_m[b, idx]: canv[b, idx] = pred[b, idx]
        return canv

class StressDualBrainModel(nn.Module):
    def __init__(self, config: MiniMindConfig, thought_dim: int = 256, pure_thought_only: bool = False):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.thought_dim = thought_dim
        self.pure_thought_only = pure_thought_only

        self.backbone = MiniMindForCausalLM(config)
        for layer in self.backbone.model.layers:
            layer.self_attn.is_causal = False

        self.domain_proj = nn.Linear(config.hidden_size, thought_dim)
        self.struct_proj = nn.Linear(config.hidden_size, thought_dim)

        nhead = 4 if thought_dim >= 32 else 2
        enc_layer = nn.TransformerEncoderLayer(
            d_model=thought_dim, nhead=nhead, dim_feedforward=thought_dim*2,
            batch_first=True, norm_first=True
        )
        self.struct_brain = nn.TransformerEncoder(enc_layer, num_layers=2, enable_nested_tensor=False)
        self.next_struct_head = nn.Linear(thought_dim, thought_dim)

        self.thought_to_prefix = nn.Linear(thought_dim * 2, config.hidden_size * 2)
        self._active_prefix = None

        def _hook(module, inp, out):
            if self._active_prefix is not None:
                out = out.clone()
                out[:, :2, :] = self._active_prefix
            return out

        self.backbone.model.embed_tokens.register_forward_hook(_hook)

    def forward_fast_muscle(self, prompt_ids, canvas_ids):
        self._active_prefix = None
        P = prompt_ids.size(1)
        full_ids = torch.cat([prompt_ids, canvas_ids], dim=1)
        return self.backbone(full_ids).logits[:, P:, :].contiguous()

    def encode_thought(self, clause_ids, valid_mask, noise_std: float = 0.0):
        self._active_prefix = None
        h = self.backbone.model(clause_ids)[0]
        mask_f = valid_mask.unsqueeze(-1).type_as(h)
        pooled = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
        d = F.normalize(self.domain_proj(pooled), p=2, dim=-1)
        s = F.normalize(self.struct_proj(pooled), p=2, dim=-1)
        if noise_std > 0.0:
            d = F.normalize(d + torch.randn_like(d) * noise_std, p=2, dim=-1)
            s = F.normalize(s + torch.randn_like(s) * noise_std, p=2, dim=-1)
        return d, s

    def forward_struct_hop(self, s_seq):
        B, T, _ = s_seq.shape
        c_mask = torch.triu(torch.ones(T, T, device=s_seq.device, dtype=torch.bool), diagonal=1)
        h = self.struct_brain(s_seq, mask=c_mask)
        return F.normalize(self.next_struct_head(h), p=2, dim=-1)

    def forward_thought_render(self, d, s, canvas_ids):
        B, L = canvas_ids.shape
        z = torch.cat([d, s], dim=-1)
        self._active_prefix = self.thought_to_prefix(z).reshape(B, 2, self.hidden_size)
        dummy_prefix = torch.full((B, 2), BOS_ID, dtype=torch.long, device=canvas_ids.device)
        full_ids = torch.cat([dummy_prefix, canvas_ids], dim=1)
        logits = self.backbone(full_ids).logits[:, 2:, :].contiguous()
        self._active_prefix = None
        return logits

    @torch.no_grad()
    def probe_muscle(self, prompt_ids, steps=2):
        B = prompt_ids.size(0)
        canv = torch.full((B, SLOT_LEN), MASK_ID, dtype=torch.long, device=prompt_ids.device)
        min_conf = 0.0
        for s in range(steps):
            logits = self.forward_fast_muscle(prompt_ids, canv)
            probs = F.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)
            if s == 0:
                non_pad = (pred != SLOT_PAD_ID)
                min_conf = conf[non_pad].min().item() if non_pad.any() else conf.min().item()
            still_m = (canv == MASK_ID)
            n_rem = still_m[0].sum().item()
            if n_rem == 0: break
            conf_m = conf.masked_fill(~still_m, -1.0)
            k = n_rem if s == steps - 1 else max(1, SLOT_LEN // steps)
            _, top_idx = torch.topk(conf_m, k=min(k, n_rem), dim=-1)
            for b in range(B):
                for idx in top_idx[b]:
                    if still_m[b, idx]: canv[b, idx] = pred[b, idx]
        return min_conf, canv

    @torch.no_grad()
    def render_from_ds(self, d, s, steps=2):
        B = d.size(0)
        canv = torch.full((B, SLOT_LEN), MASK_ID, dtype=torch.long, device=d.device)
        for step in range(steps):
            logits = self.forward_thought_render(d, s, canv)
            conf, pred = F.softmax(logits, dim=-1).max(dim=-1)
            still_m = (canv == MASK_ID)
            n_rem = still_m[0].sum().item()
            if n_rem == 0: break
            conf_m = conf.masked_fill(~still_m, -1.0)
            k = n_rem if step == steps - 1 else max(1, SLOT_LEN // steps)
            _, top_idx = torch.topk(conf_m, k=min(k, n_rem), dim=-1)
            for b in range(B):
                for idx in top_idx[b]:
                    if still_m[b, idx]: canv[b, idx] = pred[b, idx]
        return canv

    @torch.no_grad()
    def solve_multi_hop_analogy(self, c0_ids, c0_val, c1_ids, c1_val, hops: int = 1, hop_noise: float = 0.0):
        '''
        If hops=1: predicts s_2 from [s_0, s_1]
        If hops=2: predicts s_2 -> s_3 from [s_0, s_1]
        If hops=3 (Long-horizon stress!): predicts s_1 -> s_2 -> s_3 from ONLY [s_0] + law direction!
        '''
        d0, s0 = self.encode_thought(c0_ids, c0_val)
        d1, s1 = self.encode_thought(c1_ids, c1_val)
        d_tgt = F.normalize(d0 + d1, p=2, dim=-1)
        curr_seq = torch.cat([s0, s1], dim=0).unsqueeze(0)  # (1, 2, D)
        for h_i in range(hops):
            next_s = self.forward_struct_hop(curr_seq)[:, -1, :]
            if hop_noise > 0.0:
                next_s = F.normalize(next_s + torch.randn_like(next_s) * hop_noise, p=2, dim=-1)
            curr_seq = torch.cat([curr_seq, next_s.unsqueeze(1)], dim=1)
        final_s = curr_seq[:, -1, :]
        return self.render_from_ds(d_tgt, final_s, steps=2)

# Instantiate 5 competing models across scales
cfg_512 = MiniMindConfig(hidden_size=512, num_hidden_layers=8, use_moe=False)
cfg_256 = MiniMindConfig(hidden_size=256, num_hidden_layers=4, use_moe=False)
cfg_128 = MiniMindConfig(hidden_size=128, num_hidden_layers=2, use_moe=False)

torch.manual_seed(42)
m_ar = MiniMindForCausalLM(cfg_512).cuda().bfloat16()
torch.manual_seed(42)
m_diff = PureDiffusionModel(cfg_512).cuda().bfloat16()
torch.manual_seed(42)
# Pure-Thought LCM compresses all sentences (including 16 confusable serials!) into a single continuous thought vector without fast token attention!
m_lcm = StressDualBrainModel(cfg_512, thought_dim=24, pure_thought_only=True).cuda().bfloat16()
torch.manual_seed(42)
m_micro_dual = StressDualBrainModel(cfg_128, thought_dim=32, pure_thought_only=False).cuda().bfloat16()
torch.manual_seed(42)
m_tiny_dual = StressDualBrainModel(cfg_256, thought_dim=128, pure_thought_only=False).cuda().bfloat16()
torch.manual_seed(42)
m_full_dual = StressDualBrainModel(cfg_512, thought_dim=256, pure_thought_only=False).cuda().bfloat16()

def count_m(m): return sum(p.numel() for p in m.parameters()) / 1e6
print("Models instantiated:", flush=True)
for nm, md in [("AR-512", m_ar), ("Diff-512", m_diff), ("PureThought-LCM-512", m_lcm), ("Micro-Dual-128", m_micro_dual), ("Tiny-Dual-256", m_tiny_dual), ("Full-Dual-512", m_full_dual)]:
    print(f"  {nm:<22}: {count_m(md):6.2f}M params", flush=True)

# 1. Train AR-512 and Pure-Diffusion-512
ar_seqs, diff_p, diff_a = [], [], []
for pt, at, q_s, a_s in rote_tensors:
    ar_seqs.append([BOS_ID] + tokenizer(q_s + a_s, add_special_tokens=False).input_ids + [EOS_ID])
    diff_p.append(pt[0]); diff_a.append(at[0])

for d_idx, d_name, is_seen, law_id, clauses, c_t in dom_law_tensors:
    if is_seen:
        ar_seqs.append([BOS_ID] + tokenizer("".join(clauses), add_special_tokens=False).input_ids + [EOS_ID])
        diff_p.append(pad_prompt(clauses[0] + clauses[1])[0]); diff_a.append(c_t[2][0][0])
    else:
        for c in clauses:
            ar_seqs.append([BOS_ID] + tokenizer(c, add_special_tokens=False).input_ids + [EOS_ID])

opt_ar = torch.optim.AdamW(m_ar.parameters(), lr=3e-4)
opt_diff = torch.optim.AdamW(m_diff.parameters(), lr=3e-4)
t_dp = torch.stack(diff_p); t_da = torch.stack(diff_a)

for step in range(1, 161):
    opt_ar.zero_grad()
    # Sample mini-batch of 16 sequences for AR speed
    batch_s = random.sample(ar_seqs, min(16, len(ar_seqs)))
    loss_a = sum(m_ar(torch.tensor([s], device='cuda'), labels=torch.tensor([s], device='cuda')).loss for s in batch_s) / len(batch_s)
    loss_a.backward(); torch.nn.utils.clip_grad_norm_(m_ar.parameters(), 1.0); opt_ar.step()

    opt_diff.zero_grad()
    loss_d = m_diff.forward_loss(t_dp, t_da)
    loss_d.backward(); torch.nn.utils.clip_grad_norm_(m_diff.parameters(), 1.0); opt_diff.step()

# 2. Train PureThought-LCM, Micro-Dual-128, Tiny-Dual-256, Full-Dual-512
def train_stress_model(mod, steps=220):
    opt_m = torch.optim.AdamW(mod.parameters(), lr=3e-4)
    mod.train()
    # Pre-stack all 12 domain-law chains x 4 clauses = 48 clauses
    all_ct, all_cv = [], []
    for _, _, _, _, _, c_t in dom_law_tensors:
        for ct, cv in c_t:
            all_ct.append(ct[0]); all_cv.append(cv[0])
    b_ct = torch.stack(all_ct); b_cv = torch.stack(all_cv)  # (48, SLOT_LEN)

    rt_p = torch.stack([x[0][0] for x in rote_tensors])
    rt_a = torch.stack([x[1][0] for x in rote_tensors])
    rt_qt = torch.stack([pad_slot(x[2])[0][0] for x in rote_tensors])
    rt_qv = torch.stack([pad_slot(x[2])[1][0] for x in rote_tensors])

    for step in range(1, steps + 1):
        lr = 3e-5 + 0.5 * (3e-4 - 3e-5) * (1 + math.cos(math.pi * step / steps))
        for pg in opt_m.param_groups: pg['lr'] = lr
        opt_m.zero_grad()
        loss_tot = 0.0

        if not mod.pure_thought_only:
            # Train System 1 Fast Muscle Memory on Rote Serials + Emergency Codes
            m1 = (torch.rand_like(rt_a, dtype=torch.float) < 0.5); m1[:, 0] = True
            m2 = torch.ones_like(rt_a, dtype=torch.bool)
            for m in (m1, m2):
                c = rt_a.clone(); c[m] = MASK_ID
                l = rt_a.clone(); l[~m] = -100
                lg = mod.forward_fast_muscle(rt_p, c)
                loss_tot = loss_tot + F.cross_entropy(lg.reshape(-1, lg.size(-1)), l.reshape(-1), ignore_index=-100)
        else:
            # Pure Thought LCM: Must squeeze all 20 Rote Serials through continuous thought vector with realistic latent noise!
            dq_r, sq_r = mod.encode_thought(rt_qt, rt_qv, noise_std=0.04)
            canv_r = torch.full_like(rt_a, MASK_ID)
            lg_r = mod.forward_thought_render(dq_r, sq_r, canv_r)
            loss_tot = loss_tot + F.cross_entropy(lg_r.reshape(-1, lg_r.size(-1)), rt_a.reshape(-1), ignore_index=-100)

        # Encode all 48 domain-law clauses: shape (6 domains, 2 laws, 4 stages, D)
        d_vec, s_vec = mod.encode_thought(b_ct, b_cv)
        D_th = d_vec.size(-1)
        d_grid = d_vec.view(6, 2, 4, D_th)
        s_grid = s_vec.view(6, 2, 4, D_th)

        loss_dis = 0.0
        # 1. Domain invariance: all 8 clauses of domain i (across both laws & 4 stages) share d_i!
        d_anchor = d_grid[:, 0, 0, :]  # (6, D)
        loss_dis = loss_dis + (1.0 - (d_grid * d_anchor.unsqueeze(1).unsqueeze(2)).sum(dim=-1)).mean()
        # Domain orthogonality across the 6 domains
        d_gram = torch.matmul(d_anchor, d_anchor.T)
        eye_6 = torch.eye(6, device='cuda', dtype=torch.bool)
        loss_dis = loss_dis + F.relu(d_gram[~eye_6] + 0.1).mean()

        # 2. Structural isomorphism: across all 6 domains, (law_id, stage_j) shares universal structural vector s_{law, stage}!
        s_anchor = s_grid[0, :, :, :]  # (2, 4, D) from Domain 0
        loss_dis = loss_dis + (1.0 - (s_grid * s_anchor.unsqueeze(0)).sum(dim=-1)).mean()
        # Orthogonal repulsion across all 8 distinct structural states (2 laws x 4 stages = 8 states)
        s_flat8 = s_anchor.reshape(8, D_th)
        s_gram = torch.matmul(s_flat8, s_flat8.T)
        eye_8 = torch.eye(8, device='cuda', dtype=torch.bool)
        loss_dis = loss_dis + F.relu(s_gram[~eye_8] + 0.1).mean()

        # 3. Train Structural Hopping ONLY on the 2 Seen Domains (Water & Highway, both laws: 4 chains of length 4)!
        s_seen_chains = s_grid[:2, :, :, :].reshape(4, 4, D_th)
        pred_s = mod.forward_struct_hop(s_seen_chains[:, :-1, :])
        loss_hop = (1.0 - (pred_s * s_seen_chains[:, 1:, :].detach()).sum(dim=-1)).mean()

        # 4. Train Thought-to-Language Block-Diffusion Mapper on all 48 clauses
        for mask_p in (0.5, 1.0):
            m = (torch.rand_like(b_ct, dtype=torch.float) < mask_p); m[:, 0] = True
            c = b_ct.clone(); c[m] = MASK_ID
            l = b_ct.clone(); l[~m] = -100
            lg_s2 = mod.forward_thought_render(d_vec, s_vec, c)
            loss_tot = loss_tot + F.cross_entropy(lg_s2.reshape(-1, lg_s2.size(-1)), l.reshape(-1), ignore_index=-100)

        loss_tot = loss_tot + 2.5 * loss_hop + 0.8 * loss_dis
        loss_tot.backward()
        torch.nn.utils.clip_grad_norm_(mod.parameters(), 1.0)
        opt_m.step()
    mod.eval()

print("Training competing models on the 40-Task Multi-Law & High-Entropy Stress Suite...", flush=True)
train_stress_model(m_lcm, steps=220)
train_stress_model(m_micro_dual, steps=220)
train_stress_model(m_tiny_dual, steps=220)
train_stress_model(m_full_dual, steps=220)
m_ar.eval(); m_diff.eval()

# ==============================================================================
# EVALUATE ON THE 40-TASK NON-SATURATED STRESS BENCHMARK
# ==============================================================================
print("\n==================================================================================================================================", flush=True)
print("=== [NON-SATURATED STRESS BENCHMARK] 40 HARD TASKS ACROSS 4 TIERS (Exposing Capacity & Architectural Limits!) ===", flush=True)
print("==================================================================================================================================", flush=True)
print(f"{'Model Architecture':<28} | {'Params':>7} | {'Tier1:16 Serials':>16} | {'Tier2:8 MultiLaw':>16} | {'Tier3:8 LongHop':>15} | {'Tier4:8 Hybrid':>14} | {'Total (40)':>10} | {'IQ Density':>12}", flush=True)
print("-" * 134, flush=True)

for name_m, mod in [
    ("1. AR-MiniMind (8L,512D)", m_ar),
    ("2. Pure-Diffusion (8L,512D)", m_diff),
    ("3. Pure-Thought LCM (512D)", m_lcm),
    ("4. Micro Dual-Brain (2L,128D)", m_micro_dual),
    ("5. Tiny Dual-Brain (4L,256D)", m_tiny_dual),
    ("6. Full Dual-Brain (8L,512D)", m_full_dual),
]:
    t1_ok, t2_ok, t3_ok, t4_ok = 0, 0, 0, 0

    # --- Tier 1: 16 Confusable High-Entropy Serial Numbers ---
    for q_s, gt_a in rote_serials:
        if mod is m_ar:
            inp = torch.tensor([[BOS_ID] + tokenizer(q_s, add_special_tokens=False).input_ids], device='cuda')
            out = mod.generate(inp, max_new_tokens=11, do_sample=False, use_cache=True)
            pred = clean_str(out[0, inp.size(1):])
        elif mod is m_diff:
            pred = clean_str(mod.generate_slots(pad_prompt(q_s), steps=2)[0])
        elif mod.pure_thought_only:
            qt, qv = pad_slot(q_s)
            dq, sq = mod.encode_thought(qt, qv, noise_std=0.04)
            pred = clean_str(mod.render_from_ds(dq, sq, steps=2)[0])
        else:
            _, out = mod.probe_muscle(pad_prompt(q_s), steps=2)
            pred = clean_str(out[0])
        if pred == gt_a: t1_ok += 1

    # --- Tier 2: 8 Multi-Law 1-Hop Unseen Analogies (4 Unseen Domains x 2 Laws, predicting Stage 2 from Stage 0+1) ---
    # --- Tier 3: 8 Multi-Hop Long-Horizon Unseen Analogies (predicting Stage 3 after 2 autonomous hops + slight latent drift!) ---
    unseen_idx = 0
    for d_idx, d_name, is_seen, law_id, clauses, c_t in dom_law_tensors:
        if is_seen: continue
        p_str = clauses[0] + clauses[1]
        gt_stage2 = clauses[2]
        gt_stage3 = clauses[3]

        if mod is m_ar:
            inp = torch.tensor([[BOS_ID] + tokenizer(p_str, add_special_tokens=False).input_ids], device='cuda')
            out = mod.generate(inp, max_new_tokens=20, do_sample=False, use_cache=True)
            pred_s2 = clean_str(out[0, inp.size(1):])
            pred_s3 = pred_s2
        elif mod is m_diff:
            pred_s2 = clean_str(mod.generate_slots(pad_prompt(p_str), steps=2)[0])
            pred_s3 = pred_s2
        else:
            # 1-hop analogy for Stage 2
            out_s2 = mod.solve_multi_hop_analogy(c_t[0][0], c_t[0][1], c_t[1][0], c_t[1][1], hops=1, hop_noise=0.02)
            pred_s2 = clean_str(out_s2[0])
            # Multi-hop analogy for Stage 3 (with cumulative latent drift noise=0.08 per hop!)
            out_s3 = mod.solve_multi_hop_analogy(c_t[0][0], c_t[0][1], c_t[1][0], c_t[1][1], hops=2, hop_noise=0.08)
            pred_s3 = clean_str(out_s3[0])

        hit_s2 = (gt_stage2[:5] in pred_s2)
        hit_s3 = (gt_stage3[:5] in pred_s3)
        if hit_s2: t2_ok += 1
        if hit_s3: t3_ok += 1

        # --- Tier 4: 8 Hybrid Compound Tasks (Must get BOTH Unseen Analogy Stage 2 AND Domain Emergency Code!) ---
        emg_q, emg_gt = domain_emg_codes[unseen_idx // 2]
        if mod is m_ar:
            inp_e = torch.tensor([[BOS_ID] + tokenizer(emg_q, add_special_tokens=False).input_ids], device='cuda')
            pred_emg = clean_str(mod.generate(inp_e, max_new_tokens=10, do_sample=False, use_cache=True)[0, inp_e.size(1):])
        elif mod is m_diff:
            pred_emg = clean_str(mod.generate_slots(pad_prompt(emg_q), steps=2)[0])
        elif mod.pure_thought_only:
            qt, qv = pad_slot(emg_q)
            dq, sq = mod.encode_thought(qt, qv, noise_std=0.04)
            pred_emg = clean_str(mod.render_from_ds(dq, sq, steps=2)[0])
        else:
            _, out_e = mod.probe_muscle(pad_prompt(emg_q), steps=2)
            pred_emg = clean_str(out_e[0])

        if hit_s2 and (pred_emg == emg_gt):
            t4_ok += 1
        unseen_idx += 1

    tot = t1_ok + t2_ok + t3_ok + t4_ok
    pm = count_m(mod)
    dens = tot / pm
    print(f"{name_m:<28} | {pm:6.2f}M | {t1_ok:7d}/16 ({t1_ok/16*100:3.0f}%) | {t2_ok:7d}/8  ({t2_ok/8*100:3.0f}%) | {t3_ok:6d}/8  ({t3_ok/8*100:3.0f}%) | {t4_ok:5d}/8  ({t4_ok/8*100:3.0f}%) | {tot:4d}/40 ({tot/40*100:2.0f}%) | {dens:7.2f} t/M", flush=True)


Models instantiated:


  AR-512                :  30.03M params


  Diff-512              :  30.03M params


  PureThought-LCM-512   :  30.11M params


  Micro-Dual-128        :   1.31M params


...

2. Pure-Diffusion (8L,512D)  |  30.03M |       4/16 ( 25%) |       0/8  (  0%) |      0/8  (  0%) |     0/8  (  0%) |    4/40 (10%) |    0.13 t/M


3. Pure-Thought LCM (512D)   |  30.11M |       0/16 (  0%) |       6/8  ( 75%) |      7/8  ( 88%) |     3/8  ( 38%) |   16/40 (40%) |    0.53 t/M


4. Micro Dual-Brain (2L,128D) |   1.31M |       0/16 (  0%) |       0/8  (  0%) |      0/8  (  0%) |     0/8  (  0%) |    0/40 ( 0%) |    0.00 t/M


5. Tiny Dual-Brain (4L,256D) |   5.46M |       4/16 ( 25%) |       8/8  (100%) |      7/8  ( 88%) |     8/8  (100%) |   27/40 (68%) |    4.94 t/M


6. Full Dual-Brain (8L,512D) |  31.93M |       3/16 ( 19%) |       8/8  (100%) |      6/8  ( 75%) |     8/8  (100%) |   25/40 (62%) |    0.78 t/M
